# Volume of Mixing vs Pressure

Computes ΔV_mix(P*) for 11 pressures P* = 1.0 – 2.0.

## Definition

$$\Delta V_{\rm mix} = V_{\rm mix} - V_{\rm pol} - V_{\rm sol}$$

All three are **NPT-equilibrated box volumes** at the same P*, read from
`box_dimensions_*.dat`:

- **V_mix** — the isolated gel (all polymer + all solvent, `isolated_*.data`)
  re-equilibrated fully periodic under NPT (`polymer_pure` engine, `1.0_1.0`).
- **V_pol** — pure polymer (`polymer_pure`, `*_polymer_only`, `1.0_0.0`).
- **V_sol** — pure solvent (`solvent_pure`, `*_solvent_only`, `1.0_0.0`).

No trimming and no scaling: the pure boxes hold exactly the same atoms as the
mix, so $N_{\rm pol}$ and $N_{\rm sol}$ match across all three runs (scale = 1).

## Scientific basis for this pipeline (2026-06 redesign)

- **The dominant error was an inconsistent pair potential, not geometry.**
  `polymer_pure` previously equilibrated with attractive LJ (`lj/cut 2.5`) while
  the gel is WCA (`lj/cut 1.122`). Switching it to WCA cut
  ΔV_mix/(V_sol+V_pol) ~10× (≈0.20 → ≈0.02). All three boxes now use identical
  WCA (`lj/cut 1.122`, ε = 1).
- **V_mix must be the same kind of volume as the references.** It was a
  geometric bounding box (`isolate_gel.py` extent + clearance); the references
  were NPT volumes. Subtracting different volume types is not a valid ΔV_mix.
  V_mix is now an NPT box volume measured identically to V_pol and V_sol.
- **Clearance was a vertical offset, not the trend** (clearance-sensitivity
  sweep): deflating the isolate clearance 0.2 → 0 shifted every point by a
  near-constant ≈0.018 in ΔV_mix/(V_sol+V_pol) (0.0180 at P*=1.0, 0.0184 at
  P*=2.0) with unchanged slope (~0.0088 → 0.0084 per unit P*). Removing it
  cleans the absolute baseline but cannot explain the pressure dependence.
- **The remaining pressure dependence is physical packing frustration.** Bonded
  beads sit at the FENE equilibrium ≈0.97σ; nonbonded beads contact at
  2^(1/6)σ ≈ 1.122σ. This ~13 % size/spacing asymmetry produces excess volume
  that grows under compression. Matching the two lengths would destroy the
  Kremer–Grest no-chain-crossing property, so it is retained. Expect a small
  (~1–3 %), rising ΔV_mix as a model property, not a bug.
- **Trimming/scaling removed.** All polymer and all solvent seed both the pure
  boxes and the mix, eliminating the per-species percentile-trim
  density-extrapolation assumption (scale factors are identically 1).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({
    'font.family':        'CMU Serif',
    'mathtext.fontset':   'cm',
    'mathtext.rm':        'CMU Serif',
    'font.size':          20,
    'axes.titlesize':     22,
    'axes.labelsize':     25,
    'xtick.labelsize':    23,
    'ytick.labelsize':    23,
    'legend.fontsize':    23,
    'figure.titlesize':   22,
    'axes.unicode_minus': False,
    'figure.dpi':         120,
})

# --- Configuration ---
BASE_DATANAME  = "slab_support_5beads_tall_rho04"
INTERACTION    = "1.0_1.0"
PURE_INTER     = "1.0_0.0"
SLAB_STEPS     = 600000
PURE_STEPS     = 100000
N_SNAPS        = 3      # slab snapshots per pressure (volmix_sweep.sh N_SNAPS)
NREPS          = 3      # NPT thermal replicas per snapshot (volmix_sweep.sh NREPS)
# Total ΔV_mix samples per pressure: N_SNAPS × NREPS

DATA_DIR = Path("../../flow_data_local/volmix_sweep")

PRESSURES = [round(1.0 + i * 0.1, 1) for i in range(11)]
print(f"Pressures: {PRESSURES}")
print(f"Snapshots: {N_SNAPS}")
print(f"Reps/snap: {NREPS}")
print(f"Samples/P: {N_SNAPS*NREPS}")
print(f"Data root: {DATA_DIR.resolve()}")


In [ ]:
# === Sync volume data from Expanse ===
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
STAGE_DIR    = "/home/dpollard/Documents/lammps_runs/volmix_sweep/volmix_stage"

# Stage the THREE NPT box_dimensions files (mix, pure polymer, pure solvent) for
# each pressure x replica.  Files carry _rep1/_rep2/_rep3 in their names so they
# are all unique and can coexist in the same p${P}/ directory.
stage_script = (
    "SWEEP=~/Documents/lammps_runs/volmix_sweep\n"
    "DATA=~/Documents/lammps_data\n"
    "STAGE=${SWEEP}/volmix_stage\n"
    "BASE=slab_support_5beads_tall_rho04\n"
    "mkdir -p \"$STAGE\"\n"
    "for P in 1.0 1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.0; do\n"
    "  mkdir -p \"$STAGE/p${P}\"\n"
    "  for SNAP in 1 2 3; do\n"
    "    for REP in 1 2 3; do\n"
    "      ISTEM=\"isolated_${BASE}_pstar${P}_1.0_1.0_600000_snap${SNAP}\"\n"
    "      MIX_F=\"box_dimensions_${ISTEM}_rep${REP}_1.0_1.0_100000.dat\"\n"
    "      POL_F=\"box_dimensions_${ISTEM}_polymer_only_rep${REP}_1.0_0.0_100000.dat\"\n"
    "      SOL_F=\"box_dimensions_${ISTEM}_solvent_only_rep${REP}_1.0_0.0_100000.dat\"\n"
    "      for F in \"$MIX_F\" \"$POL_F\" \"$SOL_F\"; do\n"
    "        SRC=$(find \"$SWEEP\" -name \"$F\" -not -path '*/volmix_stage/*' -printf '%T@ %p\\n' 2>/dev/null | sort -rn | head -1 | cut -d' ' -f2-)\n"
    "        [ -n \"$SRC\" ] && cp \"$SRC\" \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "      done\n"
    "    done\n"
    "  done\n"
    "  cnt=$(ls \"$STAGE/p${P}/\" 2>/dev/null | wc -l)\n"
    "  echo \"  P=${P}: $cnt files staged\"\n"
    "done\n"
)

password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
totp     = getpass.getpass("TOTP / verification code: ")

def auth_handler(title, instructions, prompt_list):
    responses = []
    for prompt, echo in prompt_list:
        if "password" in prompt.strip().lower():
            responses.append(password)
        else:
            responses.append(totp)
    return responses

print("Connecting to Expanse...")
transport = paramiko.Transport((EXPANSE_HOST, 22))
transport.connect()
transport.auth_interactive(EXPANSE_USER, auth_handler)

ssh = paramiko.SSHClient()
ssh._transport = transport

print("Step 1 — staging files on Expanse...")
_, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
stdout.channel.sendall(stage_script.encode())
stdout.channel.shutdown_write()
print(stdout.read().decode())

print("Step 2 — downloading via SFTP (skips files already present)...")
sftp = ssh.open_sftp()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sftp_download_dir(sftp, remote_dir, local_dir):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = local_dir / entry.filename
        if stat.S_ISDIR(entry.st_mode):
            sftp_download_dir(sftp, remote_path, local_path)
        else:
            if local_path.exists() and local_path.stat().st_mtime >= entry.st_mtime:
                continue
            sftp.get(remote_path, str(local_path))

sftp_download_dir(sftp, STAGE_DIR, DATA_DIR)
sftp.close()
ssh.close()
print("Sync complete.")


## Volume parsing functions

In [ ]:
def avg_box_volume(path, skip_frac=0.5):
    """
    Parse box_dimensions_*.dat (columns: step lx ly lz).
    Returns time-averaged volume = mean(lx*ly*lz) over the last (1-skip_frac) fraction.
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    step, lx, ly, lz = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                    data.append(lx * ly * lz)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def avg_pure_volume(path, skip_frac=0.0):
    """
    Parse vol_pure_*.dat (columns: step press_mean vol_mean rho_mean, block averages).
    Returns mean of vol_mean column over all blocks (skip_frac=0 since runs are short).
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    vol = float(parts[2])  # vol_mean column
                    data.append(vol)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def load_manifest(path):
    """Load the volmix manifest JSON written by split_gel.py."""
    with open(path) as f:
        return json.load(f)


## Load volume data for each pressure

Uses the sweep manifest files written by pressure_sweep.sh to locate each run directory.

In [ ]:
import json

results = []
missing = []

for P in PRESSURES:
    pstr = f"{P:.1f}"
    p_dir = DATA_DIR / f"p{pstr}"

    all_rows = []
    for snap in range(1, N_SNAPS + 1):
        isolated_stem = (f"isolated_{BASE_DATANAME}_pstar{pstr}"
                         f"_{INTERACTION}_{SLAB_STEPS}_snap{snap}")
        sol_stem = f"{isolated_stem}_solvent_only"
        pol_stem = f"{isolated_stem}_polymer_only"

        for rep in range(1, NREPS + 1):
            mix_vol_file = p_dir / f"box_dimensions_{isolated_stem}_rep{rep}_{INTERACTION}_{PURE_STEPS}.dat"
            sol_vol_file = p_dir / f"box_dimensions_{sol_stem}_rep{rep}_{PURE_INTER}_{PURE_STEPS}.dat"
            pol_vol_file = p_dir / f"box_dimensions_{pol_stem}_rep{rep}_{PURE_INTER}_{PURE_STEPS}.dat"

            if not all(f.exists() for f in [mix_vol_file, sol_vol_file, pol_vol_file]):
                for label, fpath in [("mix", mix_vol_file), ("sol", sol_vol_file), ("pol", pol_vol_file)]:
                    if not fpath.exists():
                        print(f"[SKIP] P*={pstr} snap={snap} rep={rep}: missing {label} -> {fpath.name}")
                continue

            try:
                V_mix = avg_box_volume(mix_vol_file, skip_frac=0.5)
                V_pol = avg_box_volume(pol_vol_file, skip_frac=0.5)
                V_sol = avg_box_volume(sol_vol_file, skip_frac=0.5)
                all_rows.append({"snap": snap, "rep": rep,
                                 "V_mix": V_mix, "V_sol": V_sol, "V_pol": V_pol,
                                 "dV_mix": V_mix - V_pol - V_sol})
            except Exception as e:
                print(f"[ERROR] P*={pstr} snap={snap} rep={rep}: {e}")

    if not all_rows:
        missing.append(pstr)
        continue

    all_df = pd.DataFrame(all_rows)
    n      = len(all_df)
    means  = all_df[["V_mix","V_sol","V_pol","dV_mix"]].mean()
    sems   = all_df[["V_mix","V_sol","V_pol","dV_mix"]].sem() if n > 1 else              pd.Series({k: 0.0 for k in ["V_mix","V_sol","V_pol","dV_mix"]})

    results.append({
        "P":          P,
        "V_mix":      means["V_mix"],    "V_mix_sem":  sems["V_mix"],
        "V_sol":      means["V_sol"],    "V_sol_sem":  sems["V_sol"],
        "V_pol":      means["V_pol"],    "V_pol_sem":  sems["V_pol"],
        "dV_mix":     means["dV_mix"],   "dV_mix_sem": sems["dV_mix"],
        "n_total":    n,                 # total measurements = n_snaps × n_reps available
    })
    print(f"P*={pstr} ({n}/{N_SNAPS*NREPS} samples):  "
          f"V_mix={means['V_mix']:.1f}  V_sol={means['V_sol']:.1f}  "
          f"V_pol={means['V_pol']:.1f}  dV={means['dV_mix']:+.2f} \u00b1 {sems['dV_mix']:.2f}")

df = pd.DataFrame(results)
print(f"\nLoaded {len(df)}/{len(PRESSURES)} pressure points")
if missing:
    print(f"Missing: {missing}")


## Plot ΔV_mix vs P*

In [ ]:
if df.empty:
    print("No data to plot — run volmix_sweep.sh and wait for all jobs to complete.")
else:
    P          = df["P"].values
    V_mix      = df["V_mix"].values
    V_sol      = df["V_sol"].values
    V_pol      = df["V_pol"].values
    dV_mix     = df["dV_mix"].values
    V_mix_sem  = df["V_mix_sem"].values
    V_sol_sem  = df["V_sol_sem"].values
    V_pol_sem  = df["V_pol_sem"].values
    dV_mix_sem = df["dV_mix_sem"].values
    V_ref      = V_sol + V_pol

    # --- Propagate SEM: dV_mix / V_ref ---
    f     = dV_mix / V_ref
    f_sem = np.sqrt(
        (V_mix_sem / V_ref)**2 +
        ((1 + f) * V_pol_sem / V_ref)**2 +
        ((1 + f) * V_sol_sem / V_ref)**2
    )

    # --- Propagate SEM: volume fractions ---
    sol_frac     = V_sol / V_mix * 100
    sol_frac_sem = np.sqrt((V_sol_sem / V_mix)**2 + (V_sol * V_mix_sem / V_mix**2)**2) * 100
    pol_frac     = V_pol / V_mix * 100
    pol_frac_sem = np.sqrt((V_pol_sem / V_mix)**2 + (V_pol * V_mix_sem / V_mix**2)**2) * 100

    # --- rho_s / rho_s0 = V_sol / V_mix ---
    # LJ units: all masses = 1, N_sol identical in both runs.
    # rho_s  = N_sol / V_mix   (apparent solvent density in gel)
    # rho_s0 = N_sol / V_sol   (pure solvent density at same P*)
    # => ratio = V_sol / V_mix
    rho_ratio     = V_sol / V_mix
    rho_ratio_sem = np.sqrt((V_sol_sem / V_mix)**2 + (V_sol * V_mix_sem / V_mix**2)**2)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # --- Panel 1: Delta V_mix / V_ref ---
    ax = axes[0]
    ax.errorbar(P, f, yerr=f_sem, fmt="o-", color="steelblue",
                lw=2, ms=7, capsize=5, capthick=1.5, elinewidth=1.5)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"$P^*$")
    ax.set_ylabel(r"$\Delta V_{\rm mix}\;/\;(V_{\rm sol}+V_{\rm pol})$")
    ax.set_title("Volume of Mixing")
    ax.grid(True, alpha=0.3)

    # --- Panel 2: Volume fractions ---
    ax2 = axes[1]
    ax2.errorbar(P, sol_frac, yerr=sol_frac_sem, fmt="s--",
                 label=r"$V_{\rm solvent}/V_{\rm mix}$",
                 color="tomato", lw=1.5, ms=5, capsize=4, capthick=1.2, elinewidth=1.2)
    ax2.errorbar(P, pol_frac, yerr=pol_frac_sem, fmt="^--",
                 label=r"$V_{\rm polymer}/V_{\rm mix}$",
                 color="seagreen", lw=1.5, ms=5, capsize=4, capthick=1.2, elinewidth=1.2)
    ax2.set_xlabel(r"$P^*$")
    ax2.set_ylabel(r"$V_i\;/\;V_{\rm mix}\;[\%]$")
    ax2.set_title("Volume Fractions (isolated gel)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # --- Panel 3: rho_s / rho_s0 ---
    ax3 = axes[2]
    ax3.errorbar(P, rho_ratio, yerr=rho_ratio_sem, fmt="o-", color="mediumpurple",
                 lw=2, ms=7, capsize=5, capthick=1.5, elinewidth=1.5)
    ax3.set_xlabel(r"$P^*$")
    ax3.set_ylabel(r"$\rho_s \;/\; \rho_{s,0}$")
    ax3.set_title(r"Solvent Density: gel vs pure")
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_DIR = Path("../../flow_data_local/plots/volmix")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(PLOT_DIR / "volume_of_mixing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOT_DIR / 'volume_of_mixing.png'}")
    n_total = df['n_total'].values
    print(f"Samples per pressure: min={n_total.min()}, max={n_total.max()} (max={N_SNAPS*NREPS})")


## Summary table

In [ ]:
if not df.empty:
    from IPython.display import display
    display_df = df[["P", "V_mix", "V_mix_sem", "V_sol", "V_sol_sem",
                      "V_pol", "V_pol_sem", "dV_mix", "dV_mix_sem", "n_total"]].copy()
    display_df.columns = [
        "P*",
        "V_mix [σ³]", "V_mix SEM",
        "V_sol [σ³]",  "V_sol SEM",
        "V_pol [σ³]",  "V_pol SEM",
        "ΔV_mix [σ³]", "ΔV_mix SEM",
        "N samples",
    ]
    display_df = display_df.round(3)
    display(display_df)
